# Credit Risk Exploratory Analysis
### Dataset: Home Credit Default Risk (Kaggle)

Mục tiêu: ôn lại SQL + pandas nền tảng, đồng thời luyện áp dụng **Khung đặt câu hỏi domain** (So sánh → Độ lớn → Nguyên nhân → Hệ quả) vào dữ liệu tín dụng thật.

Mỗi câu hỏi bên dưới có 2 phần:
- Ô code để bạn tự viết SQL (dùng `duckdb` hoặc `sqlite3` để chạy SQL trực tiếp trên DataFrame) và/hoặc pandas
- Ô "Insight" để bạn tự trả lời bằng lời — đây là phần quan trọng nhất, không phải code

> Gợi ý: cài `duckdb` (`pip install duckdb --break-system-packages`) để chạy SQL trực tiếp trên pandas DataFrame mà không cần dựng database riêng: `duckdb.sql("SELECT ... FROM df")`.


In [5]:
import pandas as pd
import duckdb  # bỏ comment nếu dùng SQL qua duckdb

df = pd.read_csv("application_train.csv/application_train.csv")
df.shape



(307511, 122)

---
## Phần 1 — Nền tảng: GROUP BY, tỷ lệ, so sánh nhóm


### Câu 1 — Tỷ lệ vỡ nợ theo trình độ học vấn
Tính tỷ lệ vỡ nợ (default rate = trung bình của `TARGET`) theo từng nhóm `NAME_EDUCATION_TYPE`, sắp xếp giảm dần.

*Đây là câu bạn từng làm dở — sửa lại lỗi JOIN/CTE lần này.*


In [25]:
# SQL (qua duckdb) hoặc pandas groupby
df1 = df[["SK_ID_CURR", "TARGET", "CODE_GENDER", "NAME_CONTRACT_TYPE", "AMT_INCOME_TOTAL", "AMT_CREDIT","NAME_EDUCATION_TYPE" ]]
duckdb.sql("""
    with cte1 (total, "NAME_EDUCATION_TYPE") as 
    (
    select count(*), NAME_EDUCATION_TYPE as total from df1 where TARGET = 1 group by NAME_EDUCATION_TYPE
    )
    SELECT (cte1.total/count(TARGET)) *1.0 as default_rate, df1.NAME_EDUCATION_TYPE   from df1 JOIN cte1 ON cte1."NAME_EDUCATION_TYPE" = df1."NAME_EDUCATION_TYPE" group by df1."NAME_EDUCATION_TYPE", cte1.total ORDER BY default_rate DESC 

    """)



┌──────────────────────┬───────────────────────────────┐
│     default_rate     │      NAME_EDUCATION_TYPE      │
│        double        │            varchar            │
├──────────────────────┼───────────────────────────────┤
│  0.10927672955974843 │ Lower secondary               │
│  0.08939928843221562 │ Secondary / secondary special │
│  0.08484966429891992 │ Incomplete higher             │
│  0.05355115344028425 │ Higher education              │
│ 0.018292682926829267 │ Academic degree               │
└──────────────────────┴───────────────────────────────┘

**Insight (áp Khung 4 bước):** So với gì? Độ lớn thế nào? Vì sao? Ảnh hưởng gì?

_(viết câu trả lời ở đây)_

### Câu 2 — Top 10% thu nhập cao nhất nhưng vẫn vỡ nợ
Tìm nhóm khách hàng thuộc top 10% thu nhập (`AMT_INCOME_TOTAL`) cao nhất, trong đó bao nhiêu % vẫn vỡ nợ (`TARGET = 1`)?

*Dùng `NTILE(10)` (SQL) hoặc `pd.qcut`/`quantile` (pandas) — đây là chỗ bạn từng vướng.*


**Insight:** Thu nhập cao có thực sự bảo vệ khỏi vỡ nợ không? Vì sao con số này có ý nghĩa hay không?

_(viết câu trả lời ở đây)_

### Câu 3 — Tỷ lệ Credit/Income và khả năng vỡ nợ
Tạo cột `CREDIT_INCOME_RATIO = AMT_CREDIT / AMT_INCOME_TOTAL`. So sánh tỷ lệ vỡ nợ giữa nhóm có tỷ lệ này cao (trên median) và nhóm thấp (dưới median).


**Insight:** Độ lớn khác biệt giữa 2 nhóm có đáng kể không? Đây có phải yếu tố rủi ro thực sự?

_(viết câu trả lời ở đây)_

---
## Phần 2 — Đào sâu hơn: nhiều chiều, window functions, tương quan


### Câu 4 — Xếp hạng độ tuổi rủi ro nhất
Tạo nhóm tuổi (chuyển `DAYS_BIRTH` thành tuổi), dùng `RANK()`/`ROW_NUMBER()` (SQL) hoặc `rank()` (pandas) để xếp hạng các nhóm tuổi theo tỷ lệ vỡ nợ, từ cao xuống thấp.


**Insight:** Nhóm tuổi nào rủi ro nhất? Có hợp lý theo trực giác không, hay bất ngờ?

_(viết câu trả lời ở đây)_

### Câu 5 — Tình trạng việc làm (DAYS_EMPLOYED) và rủi ro
So sánh tỷ lệ vỡ nợ giữa nhóm có thời gian làm việc (`DAYS_EMPLOYED`) lâu năm và nhóm mới đi làm. Lưu ý: dataset này có giá trị bất thường (outlier) ở cột này — kiểm tra trước khi phân tích.


**Insight:** Có phát hiện outlier không? Xử lý thế nào? Kết luận về mối liên hệ này?

_(viết câu trả lời ở đây)_

### Câu 6 — Loại hợp đồng vay (NAME_CONTRACT_TYPE)
So sánh tỷ lệ vỡ nợ giữa vay tiền mặt (Cash loans) và vay trả góp (Revolving loans).


**Insight:** Chênh lệch này lớn hay nhỏ? Nguyên nhân khả dĩ là gì?

_(viết câu trả lời ở đây)_

### Câu 7 — Tình trạng hôn nhân và số con
Tỷ lệ vỡ nợ có khác nhau giữa các nhóm `NAME_FAMILY_STATUS` không? Số con (`CNT_CHILDREN`) có tương quan với tỷ lệ vỡ nợ không?


**Insight:** Đây có phải yếu tố nên đưa vào mô hình rủi ro? Vì sao (cẩn thận với yếu tố nhạy cảm/đạo đức khi dùng thực tế)?

_(viết câu trả lời ở đây)_

### Câu 8 — Điểm ngoại sinh (EXT_SOURCE_1/2/3)
Dataset có 3 cột điểm tín dụng từ nguồn ngoài (`EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`). Tính tương quan (correlation) giữa mỗi cột này với `TARGET`.


**Insight:** Cột nào tương quan mạnh nhất với vỡ nợ? Điều này gợi ý gì về việc dùng dữ liệu bên thứ 3 trong chấm điểm tín dụng?

_(viết câu trả lời ở đây)_

---
## Phần 3 — Tổng hợp & mở rộng


### Câu 9 — Kết hợp nhiều yếu tố
Chọn 2-3 yếu tố có vẻ ảnh hưởng nhất từ các câu trên, nhóm theo kết hợp của chúng (vd: học vấn × loại hợp đồng vay), tìm nhóm có tỷ lệ vỡ nợ cao nhất.


**Insight:** Nhóm rủi ro cao nhất là ai? Đây có phải phát hiện đủ rõ để đề xuất hành động (vd: siết điều kiện vay) không?

_(viết câu trả lời ở đây)_

### Câu 10 — Xu hướng hay nhiễu?
Chọn 1 phát hiện ở trên có vẻ bất ngờ. Kiểm tra: nhóm đó có đủ lớn (số lượng mẫu) để tin cậy không, hay có thể chỉ là nhiễu do mẫu nhỏ?


**Insight:** Áp dụng khái niệm thống kê "trend vs noise" — kết luận có đáng tin không?

_(viết câu trả lời ở đây)_

### Câu 11 — Nếu là chuyên viên rủi ro tín dụng
Dựa trên toàn bộ phân tích, viết 3 khuyến nghị ngắn (mỗi cái 1-2 câu) cho một ngân hàng muốn giảm tỷ lệ vỡ nợ, dựa trên các yếu tố bạn vừa tìm ra.


_(viết 3 khuyến nghị ở đây — đây chính là phần "Decision/Action" trong chuỗi giá trị Raw Data → ... → Action)_

### Câu 12 — Câu hỏi tự đặt thêm
Tự nghĩ ra 1 câu hỏi mới từ dữ liệu này mà chưa ai hỏi ở trên, dùng đúng Khung 4 bước để trả lời.


**Câu hỏi tự đặt:**

_(viết ở đây)_

**Insight:**

_(viết ở đây)_

---
## Kết luận chung

_(Tóm tắt 3-5 dòng: phát hiện quan trọng nhất, điều gì bất ngờ nhất, điều gì sẽ đưa vào README khi push lên GitHub)_
